# Phase 4: Causal Patching (Native PyTorch)
Surgically inject counterfactual hidden states at Layer 10 of Llama-3.2-3B to causally corrupt the model's reasoning.

In [ ]:
!pip install -q transformers accelerate huggingface_hub
import huggingface_hub
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    huggingface_hub.login(token=hf_token)
    print("Logged in via Kaggle Secrets!")
except Exception as e:
    print(f"Kaggle Secrets failed: {e}")
    # Paste your raw token below if secrets are not configured
    hf_token = "REPLACE_ME_WITH_YOUR_RAW_HF_TOKEN"
    huggingface_hub.login(token=hf_token)

In [ ]:
import json, torch
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
TARGET_LAYER = 10

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
model.eval()
print(f"Model loaded on: {model.device}")

In [ ]:
print("Downloading dataset...")
file_path = hf_hub_download(
    repo_id="AnishRacherla/LinguaFranca-Phase3",
    filename="data/2wikimultihopqa/augmented.jsonl",
    repo_type="dataset"
)
with open(file_path) as f:
    data = [json.loads(line) for line in f]

counterfactuals = [d for d in data if d.get("is_counterfactual", False)]
cleans = {d["id"]: d for d in data if not d.get("is_counterfactual", False)}
print(f"Loaded {len(cleans)} clean examples, {len(counterfactuals)} counterfactuals.")

pairs = []
for cf in counterfactuals:
    clean = cleans.get(cf.get("clean_pair_id"))
    if not clean:
        continue
    clean_h1 = next((h for h in clean.get("hops", []) if h["hop_idx"] == 1), None)
    cf_h1 = next((h for h in cf.get("hops", []) if h["hop_idx"] == 1), None)
    if clean_h1 and cf_h1 and clean_h1.get("label") == 0 and cf_h1.get("label") == 1:
        pairs.append((clean, cf))

print(f"Found {len(pairs)} perfect minimal pairs.")

In [ ]:
def get_layer_hidden_state(prompt, layer_idx):
    """Run a forward pass and extract the last-token hidden state at layer_idx."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True)
    # hidden_states[0] = embedding output, hidden_states[i+1] = layer i output
    hidden = out.hidden_states[layer_idx + 1]  # shape: [1, seq_len, hidden_dim]
    return hidden[0, -1, :].clone()  # last token, shape: [hidden_dim]


def generate_with_patch(prompt, patched_state, layer_idx, max_new_tokens=60):
    """Generate text with a patched hidden state injected at layer_idx."""
    
    # Flag to only inject once (on the prefill forward pass, not every decode step)
    injected = [False]

    def hook_fn(module, args, output):
        # In new HF/transformers, decoder layer output can be a tuple OR a tensor
        # Safely handle both cases
        if isinstance(output, tuple):
            hidden = output[0]  # shape: [batch, seq, hidden]
            rest = output[1:]
        else:
            hidden = output
            rest = None

        # Only inject during the prefill (seq_len > 1), not token-by-token decode
        if not injected[0] and hidden.shape[1] > 1:
            hidden = hidden.clone()
            hidden[0, -1, :] = patched_state
            injected[0] = True

        if rest is not None:
            return (hidden,) + rest
        return hidden

    handle = model.model.layers[layer_idx].register_forward_hook(hook_fn)
    try:
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out_ids = model.generate(
                inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        prompt_len = inputs.input_ids.shape[1]
        gen_ids = out_ids[0][prompt_len:]
        return tokenizer.decode(gen_ids, skip_special_tokens=True)
    finally:
        handle.remove()


print(f"Running causal patching on {len(pairs)} pairs at Layer {TARGET_LAYER}...")
successes = 0
total = len(pairs)

for i, (clean, cf) in enumerate(tqdm(pairs, desc="Brain Surgery")):
    cf_prompt = cf["prompt"]
    clean_prompt = clean["prompt"]
    substitute = cf.get("substitute", "")

    # Step 1: Harvest the hallucinating hidden state from the CF prompt
    cf_hidden = get_layer_hidden_state(cf_prompt, TARGET_LAYER)

    # Step 2: Generate with that state injected into the clean prompt
    generated = generate_with_patch(clean_prompt, cf_hidden, TARGET_LAYER)

    if substitute and substitute.lower() in generated.lower():
        successes += 1

    if i == 0:
        print("\n=== First example ===")
        print(f"Clean Q: {clean['question']}")
        print(f"CF Q:    {cf['question']}")
        print(f"Inject substitute: '{substitute}'")
        print(f"Corrupted generation:\n{generated}")
        print("======================\n")

rate = successes / total * 100 if total > 0 else 0
print(f"\n=== RESULTS ===")
print(f"Pairs tested:        {total}")
print(f"Successful corruptions: {successes}")
print(f"Causal Corruption Rate: {rate:.1f}%")